### Experiment notes — 1

Implementation details for the methods in this notebook.

*English presentation summary; the complete original narrative is preserved in `archive/original_notebook.ipynb`.*

### Experiment notes — 2

Review the implementation and results for this stage of the experiment.

*English presentation summary; the complete original narrative is preserved in `archive/original_notebook.ipynb`.*

### Experiment notes — 3

Review the implementation and results for this stage of the experiment.

*English presentation summary; the complete original narrative is preserved in `archive/original_notebook.ipynb`.*

In [89]:
from collections import Counter
import pandas as pd # data process
import numpy as np # Implementation note
from sklearn.model_selection import train_test_split, cross_validate # split dataset function
from sklearn.metrics import accuracy_score # accuracy function
import gc

RANDOM_SEED = 2020 # Implementation note

### Experiment notes — 5

Review the implementation and results for this stage of the experiment.

*English presentation summary; the complete original narrative is preserved in `archive/original_notebook.ipynb`.*

In [90]:
csv_data = './data/high_diamond_ranked_10min.csv' # data path
data_df = pd.read_csv(csv_data, sep=',') # csv file for pandasDataFrame
data_df = data_df.drop(columns='gameId') # Implementation note

### Experiment notes — 7

Dataset overview and problem setup.

*English presentation summary; the complete original narrative is preserved in `archive/original_notebook.ipynb`.*

In [91]:
print(data_df.iloc[0]) # output first data
data_df.describe() # feature statistics

blueWins                            0.0
blueWardsPlaced                    28.0
blueWardsDestroyed                  2.0
blueFirstBlood                      1.0
blueKills                           9.0
blueDeaths                          6.0
blueAssists                        11.0
blueEliteMonsters                   0.0
blueDragons                         0.0
blueHeralds                         0.0
blueTowersDestroyed                 0.0
blueTotalGold                   17210.0
blueAvgLevel                        6.6
blueTotalExperience             17039.0
blueTotalMinionsKilled            195.0
blueTotalJungleMinionsKilled       36.0
blueGoldDiff                      643.0
blueExperienceDiff                 -8.0
blueCSPerMin                       19.5
blueGoldPerMin                   1721.0
redWardsPlaced                     15.0
redWardsDestroyed                   6.0
redFirstBlood                       0.0
redKills                            6.0
redDeaths                           9.0


,blueWins,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,blueHeralds,...,redTowersDestroyed,redTotalGold,redAvgLevel,redTotalExperience,redTotalMinionsKilled,redTotalJungleMinionsKilled,redGoldDiff,redExperienceDiff,redCSPerMin,redGoldPerMin
count,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,...,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000
mean,0.499038,22.288288,2.824881,0.504808,6.183925,6.137666,6.645106,0.549954,0.361980,0.187974,...,0.043021,16489.041401,6.925316,17961.730438,217.349226,51.313088,-14.414111,33.620306,21.734923,1648.904140
std,0.500024,18.019177,2.174998,0.500002,3.011028,2.933818,4.064520,0.625527,0.480597,0.390712,...,0.216900,1490.888406,0.305311,1198.583912,21.911668,10.027885,2453.349179,1920.370438,2.191167,149.088841
min,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,11212.000000,4.800000,10465.000000,107.000000,4.000000,-11467.000000,-8348.000000,10.700000,1121.200000
25%,0.000000,14.000000,1.000000,0.000000,4.000000,4.000000,4.000000,0.000000,0.000000,0.000000,...,0.000000,15427.500000,6.800000,17209.500000,203.000000,44.000000,-1596.000000,-1212.000000,20.300000,1542.750000
50%,0.000000,16.000000,3.000000,1.000000,6.000000,6.000000,6.000000,0.000000,0.000000,0.000000,...,0.000000,16378.000000,7.000000,17974.000000,218.000000,51.000000,-14.000000,28.000000,21.800000,1637.800000
75%,1.000000,20.000000,4.000000,1.000000,8.000000,8.000000,9.000000,1.000000,1.000000,0.000000,...,0.000000,17418.500000,7.200000,18764.500000,233.000000,57.000000,1585.500000,1290.500000,23.300000,1741.850000
max,1.000000,250.000000,27.000000,1.000000,22.000000,22.000000,29.000000,2.000000,1.000000,1.000000,...,2.000000,22732.000000,8.200000,22269.000000,289.000000,92.000000,10830.000000,9333.000000,28.900000,2273.200000


### Experiment notes — 9

Inspect and engineer input features.

*English presentation summary; the complete original narrative is preserved in `archive/original_notebook.ipynb`.*

In [92]:
drop_features = ['blueGoldDiff', 'redGoldDiff', 
                 'blueExperienceDiff', 'redExperienceDiff', 
                 'blueCSPerMin', 'redCSPerMin', 
                 'blueGoldPerMin', 'redGoldPerMin'] # feature
df = data_df.drop(columns=drop_features) # feature

In [93]:
info_names = [c[3:] for c in df.columns if c.startswith('red')] # take value feature （red
for info in info_names: # each feature
    df['br' + info] = df['blue' + info] - df['red' + info] # one feature ， feature feature ， for br
# FirstBlood for one only ，brFirstBlood=1 for ，0 for ，-1 for
df = df.drop(columns=['blueFirstBlood', 'redFirstBlood']) # FirstBlood delete

In [94]:
# process data
print(df.head(5))
df.describe()

   blueWins  blueWardsPlaced  blueWardsDestroyed  blueKills  blueDeaths  \
0         0               28                   2          9           6   
1         0               12                   1          5           5   
2         0               15                   0          7          11   
3         0               43                   1          4           5   
4         0               75                   4          6           6   

   blueAssists  blueEliteMonsters  blueDragons  blueHeralds  \
0           11                  0            0            0   
1            5                  0            0            0   
2            4                  1            1            0   
3            5                  1            0            1   
4            6                  0            0            0   

   blueTowersDestroyed  ...  brAssists  brEliteMonsters  brDragons  brHeralds  \
0                    0  ...          3                0          0          0   
1       

,blueWins,blueWardsPlaced,blueWardsDestroyed,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,blueHeralds,blueTowersDestroyed,...,brAssists,brEliteMonsters,brDragons,brHeralds,brTowersDestroyed,brTotalGold,brAvgLevel,brTotalExperience,brTotalMinionsKilled,brTotalJungleMinionsKilled
count,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,...,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000
mean,0.499038,22.288288,2.824881,6.183925,6.137666,6.645106,0.549954,0.361980,0.187974,0.051422,...,-0.017006,-0.023180,-0.051119,0.027938,0.008402,14.414111,-0.009313,-33.620306,-0.649661,-0.803421
std,0.500024,18.019177,2.174998,3.011028,2.933818,4.064520,0.625527,0.480597,0.390712,0.244369,...,5.766787,1.067934,0.878945,0.589292,0.324835,2453.349179,0.478434,1920.370438,30.942675,14.274733
min,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-25.000000,-2.000000,-1.000000,-1.000000,-2.000000,-10830.000000,-2.600000,-9333.000000,-120.000000,-72.000000
25%,0.000000,14.000000,1.000000,4.000000,4.000000,4.000000,0.000000,0.000000,0.000000,0.000000,...,-4.000000,-1.000000,-1.000000,0.000000,0.000000,-1585.500000,-0.400000,-1290.500000,-21.000000,-10.000000
50%,0.000000,16.000000,3.000000,6.000000,6.000000,6.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,14.000000,0.000000,-28.000000,-1.000000,0.000000
75%,1.000000,20.000000,4.000000,8.000000,8.000000,9.000000,1.000000,1.000000,0.000000,0.000000,...,4.000000,1.000000,1.000000,0.000000,0.000000,1596.000000,0.200000,1212.000000,20.000000,8.000000
max,1.000000,250.000000,27.000000,22.000000,22.000000,29.000000,2.000000,1.000000,1.000000,4.000000,...,23.000000,2.000000,1.000000,1.000000,4.000000,11467.000000,2.400000,8348.000000,127.000000,64.000000


### Experiment notes — 13

Inspect and engineer input features.

*English presentation summary; the complete original narrative is preserved in `archive/original_notebook.ipynb`.*

In [95]:
discrete_df = df.copy() # one data

# parameters
n_bins = 5         # Implementation note
max_unique = 10    # feature take value

for c in df.columns[1:]: # iterate one feature ， label
    '''
    请离散化每一列特征，即discrete_df[c] = ...
    
    提示：
    对于有些特征本身取值就很少，可以跳过即 if ... : continue
    对于其他特征，可以使用等区间离散化、等密度离散化或一些其他离散化方法
    可参考使用pandas.cut或qcut
    '''

    if len(df[c].unique()) <= max_unique:
        continue

    # qcut, all value use ， use cut
    try:
        discrete_df[c] = pd.qcut(discrete_df[c], q=n_bins, duplicates='drop')
    except Exception as e:
        discrete_df[c] = pd.qcut(discrete_df[c], q=n_bins)

print(discrete_df.head())

   blueWins blueWardsPlaced blueWardsDestroyed      blueKills   blueDeaths  \
0         0   (22.4, 250.0]         (1.0, 2.0]     (7.0, 9.0]   (5.0, 7.0]   
1         0   (4.999, 14.0]      (-0.001, 1.0]     (4.0, 5.0]   (4.0, 5.0]   
2         0    (14.0, 16.0]      (-0.001, 1.0]     (5.0, 7.0]  (9.0, 22.0]   
3         0   (22.4, 250.0]      (-0.001, 1.0]  (-0.001, 4.0]   (4.0, 5.0]   
4         0   (22.4, 250.0]         (3.0, 4.0]     (5.0, 7.0]   (5.0, 7.0]   

    blueAssists  blueEliteMonsters  blueDragons  blueHeralds  \
0  (10.0, 29.0]                  0            0            0   
1    (3.0, 5.0]                  0            0            0   
2    (3.0, 5.0]                  1            1            0   
3    (3.0, 5.0]                  1            0            1   
4    (5.0, 7.0]                  0            0            0   

   blueTowersDestroyed  ...        brAssists brEliteMonsters brDragons  \
0                    0  ...       (1.0, 5.0]               0         0  

### Experiment notes — 15

Dataset provenance, format and train/test split.

*English presentation summary; the complete original narrative is preserved in `archive/original_notebook.ipynb`.*

In [96]:
all_y = discrete_df['blueWins'].values # all label data
feature_names = discrete_df.columns[1:] # all feature
all_x = discrete_df[feature_names].values # all feature value ，pandasDataFrame.values take for numpyarray matrix

# split training set and test set
x_train, x_test, y_train, y_test = train_test_split(all_x, all_y, test_size=0.2, random_state=RANDOM_SEED)
all_y.shape, all_x.shape, x_train.shape, x_test.shape, y_train.shape, y_test.shape # output data

((9879,), (9879, 43), (7903, 43), (1976, 43), (7903,), (1976,))

### Experiment notes — 17

Implementation details for the methods in this notebook.

*English presentation summary; the complete original narrative is preserved in `archive/original_notebook.ipynb`.*

In [97]:
# define tree
class DecisionTree(object):
    def __init__(self, classes, features, 
                 max_depth=10, min_samples_split=10,
                 impurity_t='entropy'):
        '''
        传入一些可能用到的模型参数，也可能不会用到
        classes表示模型分类总共有几类
        features是每个特征的名字，也方便查询总的共特征数
        max_depth表示构建决策树时的最大深度
        min_samples_split表示构建决策树分裂节点时，如果到达该节点的样本数小于该值则不再分裂
        impurity_t表示计算混杂度（不纯度）的计算方式，例如entropy或gini
        '''  
        self.classes = classes
        self.features = features
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.impurity_t = impurity_t
        self.root = None # define ， training for
        
    '''
    请实现决策树算法，使得fit函数和predict函数可以正常调用，跑通之后的测试代码，
    要求之后测试代码输出的准确率大于0.6。
    
    提示：
    可以定义额外一些函数，例如
    impurity()用来计算混杂度
    gain()调用impurity用来计算信息增益
    expand_node()训练时递归函数分裂节点，考虑不同情况
        1. 无需分裂 或 达到分裂阈值
        2. 调用gain()找到最佳分裂特征，递归调用expand_node
        3. 找不到有用的分裂特征
        fit函数调用该函数返回根节点
    traverse_node()预测时遍历节点，考虑不同情况
        1. 已经到达叶节点，则返回分类结果
        2. 该特征取值在训练集中未出现过
        3. 依据特征取值进入相应子节点，递归调用traverse_node
    当然也可以有其他实现方式。

    '''
    # 1. compute 、 information gain function
    def impurity(self, y): 
        '''
        计算给定标签列表的混杂度(熵或基尼系数)
        输入:entropy / gini
        返回（混乱度）
        '''
        total = len(y)
        if total == 0:
            return 0
        counts = Counter(y)   # statistics labels table each different label
        probs = [c/total for c in counts.values()]  # compute different class
        
        if self.impurity_t == 'entropy':
            return -sum(p * np.log2(p) for p in probs)

        if self.impurity_t == 'gini':
            return 1 - sum(p * p for p in probs)
            
    def info_gain(self, y_parent, y_left, y_right):
        '''
        计算二分裂后的信息增益
        输入：
        y_left：分裂后左子节点的标签列表
        y_right：分裂后右子节点的标签列表
        y_parent：分裂前父节点的标签列表
        返回（信息增益gain）
        '''
    
        n = len(y_parent)
        n_left, n_right = len(y_left), len(y_right)
        if n_left == 0 and n_right == 0:
            return 0
        gain = self.impurity(y_parent) \
               - (n_left / n) * self.impurity(y_left) \
               - (n_right / n) * self.impurity(y_right)

        return gain


    # 2. training function split ， different , define best_feat(),check_stop_conditions() function
    def best_feat(self, X, y, features):
        '''
        在所有可用特征中寻找节点的最佳分裂点
        返回: (best_feat, best_threshold, best_gain)
        若无有效分裂，返回 (None, None, 0)
        '''
        
        best_feat = None
        best_thr = None
        best_gain = -1

        for feat in features:
            X_col = X[:, feat]

            if len(X_col) > 0 and hasattr(X_col[0], 'left') and hasattr(X_col[0], 'right'):
                X_col = np.array([(x.left + x.right) / 2 for x in X_col])

            sorted_data = sorted(zip(X_col, y), key=lambda x: x[0]) # feature value sort ， label
            thresholds = []
            for i in range(len(sorted_data)-1):
                val_i, label_i = sorted_data[i]
                val_j, label_j = sorted_data[i+1]
                if label_i != label_j:   # select different value
                    thresholds.append((val_i + val_j) / 2)

            thresholds = list(set(thresholds)) # Implementation note

            for thr in thresholds:
                left_mask = X_col <= thr
                right_mask = ~left_mask
                y_left = y[left_mask]
                y_right = y[right_mask]
                gain = self.info_gain(y, y_left, y_right)
                if gain > best_gain:
                    best_gain = gain
                    best_feat = feat
                    best_thr = thr

        return best_feat, best_thr, best_gain

    def check_stop_conditions(self, X, y, depth, features):
        '''
        判断当前节点是否需要停止分裂
        返回: (should_stop, leaf_value)
        leaf_value 仅在停止时有效，为多数类
        '''

        # 1： sample
        if len(y) < self.min_samples_split:
            majority_class = Counter(y).most_common(1)[0][0]  # class
            return True, majority_class
        
        # 2： all sample output class
        if len(np.unique(y)) == 1:
            return True, y[0]

        # 3： all sample input feature
        if X.shape[0] > 0 and np.all(X == X[0]):
            majority_class = Counter(y).most_common(1)[0][0]
            return True, majority_class

        # 4： feature （ feature split
        if len(features) == 0:
            majority_class = Counter(y).most_common(1)[0][0]
            return True, majority_class

        # 5： to
        if depth >= self.max_depth:
            majority_class = Counter(y).most_common(1)[0][0]
            return True, majority_class

        # split
        return False, None
            
    
    def expand_node(self, X, y, depth, features):
        '''
        递归构建决策树节点
        处理情况：
        1. 检查停止条件 → 返回叶节点
        2. 找到最佳分裂特征 → 划分数据集，递归构建子树
        返回（节点字典：{ 'is_leaf': bool, 'value': 类别(叶子), / 
                        'feat': 特征索引, 'thr': 阈值, /
                        'left': 左子树, 'right': 右子树 }
        3. 找不到有效分裂（增益≤0）→ 返回叶节点（多数类）
        '''

        # first ： check
        # check
        stop, leaf_val = self.check_stop_conditions(X, y, depth, features)
        if stop:
            return {'is_leaf': True, 'value': leaf_val}

        # second ： select best split feature
        best_feat, best_thr, best_gain = self.best_feat(X, y, features)

        # to split （ for 0
        if best_feat is None or best_gain <= 0:
            return {'is_leaf': True, 'value': Counter(y).most_common(1)[0][0]}

        # third ： feature take value split dataset ， build tree
        # split data
        feat_vals = X[:, best_feat]
        if len(feat_vals) > 0 and hasattr(feat_vals[0], 'left') and hasattr(feat_vals[0], 'right'):
            feat_vals = np.array([(v.left + v.right) / 2 for v in feat_vals])
        
        left_mask = feat_vals <= best_thr
        right_mask = ~left_mask
        X_left, y_left = X[left_mask], y[left_mask]
        X_right, y_right = X[right_mask], y[right_mask]

        # build tree （ feature ， for feature use
        left_subtree = self.expand_node(X_left, y_left, depth+1, features)
        right_subtree = self.expand_node(X_right, y_right, depth+1, features)
        
        return {
            'is_leaf': False,
            'feat': best_feat,
            'thr': best_thr,
            'left': left_subtree,
            'right': right_subtree,
            'value': Counter(y).most_common(1)[0][0]   # record ， prediction process （ to
        }
    
    # fit
    def fit(self, feature, label):
        assert len(self.features) == len(feature[0]) # input data feature count and model define feature count
        '''
        训练模型
        feature为二维numpy（n*m）数组，每行表示一个样本，有m个特征
        label为一维numpy（n）数组，表示每个样本的分类标签
        
        提示：一种可能的实现方式为
        self.root = self.expand_node(feature, label, depth=1) # from split ， model record
        '''
        avail_feats = list(range(feature.shape[1]))
        self.root = self.expand_node(feature, label, depth=1, features=list(range(feature.shape[1])))     

    # 3.traverse_node() training function split ， different
    def traverse_node(self, node, x):
        '''
        根据训练好的树结构对单个样本 x 进行预测
        处理情况：
        1. 到达叶节点 → 返回该节点的类别
        2. 特征取值在训练中未出现过（对于连续特征仍可通过阈值比较处理）
        3. 依据特征值进入对应子节点（递归）
        '''
        # 1： to
        if node['is_leaf']:
            return node['value']
        feat_val = x[node['feat']]
        if hasattr(feat_val, 'left') and hasattr(feat_val, 'right'):
            feat_val = (feat_val.left + feat_val.right) / 2

        # 2 & 3： threshold
        # feature ， value and threshold compare ，“”
        if feat_val <= node['thr']:
            return self.traverse_node(node['left'], x)
        else:
            return self.traverse_node(node['right'], x)

    # 4. prediction
    def predict(self, feature):
        assert len(feature.shape) in (1, 2) # only 1 or 2
        '''
        预测
        输入feature可以是一个一维numpy数组也可以是一个二维numpy数组
        如果是一维numpy（m）数组则是一个样本，包含m个特征，返回一个类别值
        如果是二维numpy（n*m）数组则表示n个样本，每个样本包含m个特征，返回一个numpy一维数组
        
        提示：一种可能的实现方式为
        if len(feature.shape) == 1: # one sample
            return self.traverse_node(self.root, feature) # from
        return np.array([self.traverse_node(self.root, f) for f in feature]) # sample
        '''
        if len(feature.shape) == 1: # one sample
            return self.traverse_node(self.root, feature)
        return np.array([self.traverse_node(self.root, f) for f in feature])

In [98]:
# define tree model ， parameters
DT = DecisionTree(classes=[0,1], features=feature_names, max_depth=5, min_samples_split=10, impurity_t='gini')

DT.fit(x_train, y_train) # training set training
p_test = DT.predict(x_test) # test set prediction ， predicted value
print(p_test) # output predicted value

# evaluate accuracy
test_acc = accuracy_score(y_test, p_test) # testing predicted value and test set label comparison accuracy
print('accuracy: {:.4f}'.format(test_acc)) # output accuracy

[0 1 0 ... 0 1 1]
accuracy: 0.7282


### Experiment notes — 20

Review the implementation and results for this stage of the experiment.

*English presentation summary; the complete original narrative is preserved in `archive/original_notebook.ipynb`.*

### Experiment notes — 21

Review the implementation and results for this stage of the experiment.

*English presentation summary; the complete original narrative is preserved in `archive/original_notebook.ipynb`.*

In [61]:
gc.collect()

0

In [60]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

# define parameters
param_grid = {
    'max_depth': [3, 5, 7, 10, 15],
    'min_samples_split': [2, 5, 10, 20],
    'impurity_t': ['gini', 'entropy']  
}

# Implementation note
best_score = 0
best_params = {}

# define parameters
max_depth_list = [3, 5, 7, 10, 15]
min_samples_split_list = [2, 5, 10, 20]
impurity_t_list = ['gini', 'entropy']

total_combos = len(max_depth_list) * len(min_samples_split_list) * len(impurity_t_list)
current = 0

for max_depth in max_depth_list:
    for min_samples_split in min_samples_split_list:
        for impurity_t in impurity_t_list:
            current += 1
            print(f'[{current}/{total_combos}] 正在测试：max_depth={max_depth}, min_samples_split={min_samples_split}, impurity_t={impurity_t}')

            dt = DecisionTree(classes=[0,1], features=feature_names,
                              max_depth=max_depth,
                              min_samples_split=min_samples_split,
                              impurity_t=impurity_t)
            
            # 5 cross-validation
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            fold_scores = []
            for train_idx, val_idx in skf.split(x_train, y_train):
                X_tr, X_val = x_train[train_idx], x_train[val_idx]
                y_tr, y_val = y_train[train_idx], y_train[val_idx]
                dt.fit(X_tr, y_tr)
                pred = dt.predict(X_val)
                fold_scores.append(accuracy_score(y_val, pred))

            mean_score = np.mean(fold_scores)
            print(f'    平均准确率: {mean_score:.4f}')
            
            if mean_score > best_score:
                best_score = mean_score
                best_params = {'max_depth': max_depth,
                               'min_samples_split': min_samples_split,
                               'impurity_t': impurity_t}
                print(f'    *** 新的最佳参数: {best_params}, 得分: {best_score:.4f}')

print("\n网格搜索完成！")
print("Best CV score:", best_score)
print("Best params:", best_params)

[1/40] 正在测试：max_depth=3, min_samples_split=2, impurity_t=gini
    平均准确率: 0.7173
    *** 新的最佳参数: {'max_depth': 3, 'min_samples_split': 2, 'impurity_t': 'gini'}, 得分: 0.7173
[2/40] 正在测试：max_depth=3, min_samples_split=2, impurity_t=entropy
    平均准确率: 0.7173
[3/40] 正在测试：max_depth=3, min_samples_split=5, impurity_t=gini
    平均准确率: 0.7173
[4/40] 正在测试：max_depth=3, min_samples_split=5, impurity_t=entropy
    平均准确率: 0.7173
[5/40] 正在测试：max_depth=3, min_samples_split=10, impurity_t=gini
    平均准确率: 0.7173
[6/40] 正在测试：max_depth=3, min_samples_split=10, impurity_t=entropy
    平均准确率: 0.7173
[7/40] 正在测试：max_depth=3, min_samples_split=20, impurity_t=gini
    平均准确率: 0.7173
[8/40] 正在测试：max_depth=3, min_samples_split=20, impurity_t=entropy
    平均准确率: 0.7173
[9/40] 正在测试：max_depth=5, min_samples_split=2, impurity_t=gini
    平均准确率: 0.7257
    *** 新的最佳参数: {'max_depth': 5, 'min_samples_split': 2, 'impurity_t': 'gini'}, 得分: 0.7257
[10/40] 正在测试：max_depth=5, min_samples_split=2, impurity_t=entropy
    平均准确率: 0.726

### Experiment notes — 24

Interpret the reported empirical results.

*English presentation summary; the complete original narrative is preserved in `archive/original_notebook.ipynb`.*

In [99]:
# define tree model ， parameters
optimal_DT = DecisionTree(classes=[0,1], features=feature_names, max_depth=5, min_samples_split=2, impurity_t='entropy')

optimal_DT.fit(x_train, y_train) # training set training
p_test = optimal_DT.predict(x_test) # test set prediction ， predicted value
print(p_test) # output predicted value

# evaluate accuracy
test_acc = accuracy_score(y_test, p_test) # testing predicted value and test set label comparison accuracy
print('accuracy: {:.4f}'.format(test_acc)) # output accuracy

[0 1 0 ... 0 1 1]
accuracy: 0.7242


### Experiment notes — 26

Review the implementation and results for this stage of the experiment.

*English presentation summary; the complete original narrative is preserved in `archive/original_notebook.ipynb`.*